# Geography
1. Create a schema
2. Remove top 3 rows and Load CSV
3. Create ZipCountry text column
4. Write to Delta table


## 1. Create the Schema

In [1]:
from pyspark.sql.types import StructType, StructField, StringType

# Define the schema
geoschema = StructType([
    StructField("Zip", StringType(), True),      #  nullable
    StructField("City", StringType(), True),     # Nullable
    StructField("State", StringType(), True),    # Nullable
    StructField("Region", StringType(), True),   # Nullable
    StructField("District", StringType(), True), # Nullable
    StructField("Country", StringType(), True)  #  nullable
])


StatementMeta(, 561f2247-e089-4284-acc2-48640981d22b, 3, Finished, Available, Finished)

## 2. Drop top 3 rows and Load CSV with schema

In [2]:
from pyspark.sql.functions import col

# Step 1: Read the CSV as a raw DataFrame (treat everything as string initially)
df_raw = spark.read.format("csv").option("header", "false").load("Files/USSales/geo.csv")

# Step 2: Drop the first 3 rows
# Lambda is a python thing for creating an in-line expression you'll only use once
df_trimmed = df_raw.rdd.zipWithIndex().filter(lambda row: row[1] >= 3).map(lambda row: row[0]).toDF()

# Step 3: Extract the new header from the remaining data
new_header = df_trimmed.first()  # The fourth row becomes the new header
df_trimmed = df_trimmed.toDF(*new_header)  # Rename columns using the new header

# Step 4: Read the CSV again, this time enforcing the correct schema
df = df_trimmed.select([col(c).cast(t.dataType) for c, t in zip(df_trimmed.columns, geoschema)])

# Step 5: Display the cleaned DataFrame
display(df)


StatementMeta(, 561f2247-e089-4284-acc2-48640981d22b, 4, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 4e8bca57-0b43-4032-b9e3-f4bf1ba556b9)

## 3. Add the ZipCountry column

In [3]:
# Adding the ZipCountry text column

from pyspark.sql.functions import concat_ws

df = df.withColumn("ZipCountry", concat_ws(",", df["Zip"], df["Country"]))

StatementMeta(, 561f2247-e089-4284-acc2-48640981d22b, 5, Finished, Available, Finished)

In [4]:
# Verifying the results of the new column

df.select("Zip", "Country", "ZipCountry").show(5)


StatementMeta(, 561f2247-e089-4284-acc2-48640981d22b, 6, Finished, Available, Finished)

+-----+-------+-----------+
|  Zip|Country| ZipCountry|
+-----+-------+-----------+
|  Zip|Country|Zip,Country|
|22654|    USA|  22654,USA|
|22655|    USA|  22655,USA|
|22656|    USA|  22656,USA|
|22657|    USA|  22657,USA|
+-----+-------+-----------+
only showing top 5 rows



## 4. Write the Delta table

In [5]:
# Let's write the Delta table

df.write.format("delta").mode("overwrite").saveAsTable("Geography")

StatementMeta(, 561f2247-e089-4284-acc2-48640981d22b, 7, Finished, Available, Finished)